In [1]:
import pandas as pd


## Implementing the retrieval metrics


In [2]:
def precision(k:int, q_rels:list, q_ret:list) -> float:
    cpt = 0
    for doc in q_ret[:k]:
        if doc in q_rels:
            cpt += 1
    return cpt / k

def recall(k:int, q_rels:list, q_ret:list) -> float:
    cpt = 0
    for doc in q_ret[:k]:
        if doc in q_rels:
            cpt += 1
    return cpt / len(q_rels) if q_rels else 0

## Connect to the VDB

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import FastEmbedSparse, QdrantVectorStore, RetrievalMode
from sentence_transformers import CrossEncoder

QDRANT_URL = "http://localhost:6333"
COLLECTION_NAME = "hybrid_collection"
N_CANDIDATES = 20

# Loaded ONCE
dense_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
sparse_embeddings = FastEmbedSparse(model_name="Qdrant/bm25")
reranker = CrossEncoder("cross-encoder/mmarco-mMiniLMv2-L12-H384-v1")

stores = {
    mode: QdrantVectorStore.from_existing_collection(
        embedding=dense_embeddings,
        sparse_embedding=sparse_embeddings,
        retrieval_mode=mode,
        url=QDRANT_URL,
        collection_name=COLLECTION_NAME,
    )
    for mode in [RetrievalMode.DENSE, RetrievalMode.SPARSE, RetrievalMode.HYBRID]
}


def retrieve_relevant_chunks(query: str, k: int, retrieval_mode: RetrievalMode = RetrievalMode.HYBRID,
                             use_reranker: bool = False) -> list:
    documents = stores[retrieval_mode].similarity_search(query, k=N_CANDIDATES)

    if use_reranker:
        scores = reranker.predict([(query, doc.page_content) for doc in documents])
        ranked = sorted(zip(documents, scores), key=lambda x: x[1], reverse=True)
        documents = [doc for doc, _ in ranked]

    return [doc.id or doc.metadata.get("_id") for doc in documents[:k]]

In [7]:
import json

benchmark_questions = pd.read_json("../questions.jsonl", lines=True)
benchmark_questions.index = benchmark_questions.query_id

with open("../qrels.json", "r", encoding="utf-8") as file:
    qrels = json.load(file)

In [8]:
K = 5

question_ids = benchmark_questions.index.tolist()
number_of_questions = len(question_ids)

retrieval_strategies = [RetrievalMode.DENSE, RetrievalMode.SPARSE, RetrievalMode.HYBRID]
reranking_strategies = [True, False]

results = {}

for use_reranker in reranking_strategies:
    average_precision = 0
    average_recall = 0
    
    print(f"{5 * '='} Reranking strategy: {use_reranker} {5 * '='}")
    for retrieval_strategy in retrieval_strategies:
        
        print(f"{5 * '='} Retrieval strategy: {retrieval_strategy} {5 * '='}")
        for q_id in question_ids:

            query = benchmark_questions.loc[q_id].question
            q_rels = qrels[q_id]

            q_ret = retrieve_relevant_chunks(query, K, retrieval_mode=retrieval_strategy, use_reranker=use_reranker)

            precisionAtK = precision(K, q_rels, q_ret)
            recallAtK = recall(K, q_rels, q_ret)


            average_precision += precisionAtK
            average_recall += recallAtK
            
        results[(retrieval_strategy, use_reranker)] = {
            "precision": average_precision / number_of_questions,
            "recall": average_recall / number_of_questions
        }
        print(f' precision : {average_precision / number_of_questions}, recall :  {average_recall / number_of_questions}')
        print(f'end={5 * "="}')
retrieval_evaluation = pd.DataFrame.from_dict(results, orient='index')

===== Reranking strategy: True =====
===== Retrieval strategy: RetrievalMode.DENSE =====
 precision : 0.1820895522388059, recall :  0.7139303482587064
end======
===== Retrieval strategy: RetrievalMode.SPARSE =====
 precision : 0.4059701492537307, recall :  1.5671641791044777
end======
===== Retrieval strategy: RetrievalMode.HYBRID =====
 precision : 0.6268656716417916, recall :  2.4129353233830844
end======
===== Reranking strategy: False =====
===== Retrieval strategy: RetrievalMode.DENSE =====
 precision : 0.12835820895522393, recall :  0.4875621890547264
end======
===== Retrieval strategy: RetrievalMode.SPARSE =====
 precision : 0.34029850746268614, recall :  1.300995024875622
end======
===== Retrieval strategy: RetrievalMode.HYBRID =====
 precision : 0.5313432835820892, recall :  2.04726368159204
end======


In [6]:
retrieval_evaluation

,,precision,recall
RetrievalMode.DENSE,True,0.182090,0.713930
RetrievalMode.SPARSE,True,0.405970,1.567164
RetrievalMode.HYBRID,True,0.626866,2.412935
RetrievalMode.DENSE,False,0.128358,0.487562
RetrievalMode.SPARSE,False,0.340299,1.300995
RetrievalMode.HYBRID,False,0.534328,2.054726
